In [1]:
import joblib
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

## load pkl file 

In [2]:
chunks = joblib.load("chunks.pkl")
index = faiss.read_index("faiss_index.index")

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## using the groq api key

In [10]:
client = Groq(api_key="") # here using api key 

## function for question embedding, retrival, prompt and response

In [5]:
def answer_question(question, k = 3):
    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    faiss.normalize_L2(question_embedding)

    distances, indices = index.search(question_embedding, k=k)

    retrieved = [chunks[i] for i in indices[0] if i != -1]
    if not retrieved:
        return "I couldn't find anything relevant in the document to answer that."

    context = "\n\n".join(retrieved)

    prompt = f"""
                You are a Python tutor.

                Answer only using the context.

                Context:
                {context}

                Question:
                {question}

                Answer:
            """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


## ask question from user

In [8]:
question = input("Ask Question: ")
question

'file handling'

## get response from model

In [9]:
print(answer_question(question))

To handle files in Python, you can open a file using the `open()` function, which returns a file object. The file object provides methods for working with the file. 

To write to a file, you can use the `write()` method, which takes a string as an argument. If the file does not exist, a new one is created. If the file already exists, opening it in write mode ('w') clears out the old data and starts fresh.

Example:
```python
fout = open('output.txt', 'w')
fout.write('This is the first line.\n')
fout.write('This is the second line.\n')
fout.close()
```
It's essential to close the file when you're done writing to it using the `close()` method. If you don't close the file, it gets closed automatically when the program ends.

Note that the `write()` method requires a string argument, so if you want to write other types of data to a file, you need to convert them to strings first.
